[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/054.%20The%20Production%20Order%20Quantity%20%28POQ%29%20Problem/P54-Tier-3_executed.ipynb)

---


# 54. The Production Order Quantity (POQ) Problem

## Tier 3: A Genetic Algorithm for Multi-Product Scheduling

This notebook represents a production schedule as a chromosome and uses a genetic algorithm to search for schedules with low setup, inventory-holding, shortage, and capacity-violation costs. A chromosome contains one production quantity for each product and period. The algorithm repeatedly selects schedules, combines them through crossover, perturbs them through mutation, and retains elite schedules.

### Goals and inputs

The example has three products and six planning periods. Its inputs are period-by-period demand, product-specific setup and holding costs, production rates, and a shared capacity of eight production hours per period. The algorithm's population size, number of generations, crossover probability, mutation probability, and elitism count are specified in the run cell.

The outputs include the best schedule found by the search, its inventory and setup pattern, a breakdown of its modeled costs and penalties, population histories, visualizations, and a comparison with a simple every-other-period production heuristic.

### Model assumptions and scope

- Demand is known and must be met by the end of the planning horizon. Shortages during the horizon are allowed in the search but incur a penalty.
- Inventory is measured at the end of each period. Positive inventory incurs a per-unit, per-period holding cost; negative inventory represents unmet cumulative demand and incurs a shortage penalty.
- A setup occurs whenever a product has positive production in a period. Setup time is not included in the capacity calculation.
- Production time is production quantity divided by the product's production rate. The period capacity is shared across products.
- Capacity violations are penalized in the fitness function; they are not prevented by a feasibility-preserving representation. Consequently, the genetic algorithm can return an infeasible schedule, and its best schedule is not a proof of a feasible or globally optimal solution.
- Production quantities are continuous-valued during the search. The schedule display rounds quantities for readability, but cost and capacity calculations use the unrounded values.
- The model does not include lead times, backlogging costs beyond the stated shortage penalty, setup times, demand uncertainty, or changes in production rates.

### Capacity feasibility check

The stated capacity and demand data do not permit a schedule that meets all demand within the horizon while respecting the shared capacity. Total required production time is

$$
\sum_{i=1}^{3}\sum_{t=1}^{6}\frac{d_{it}}{r_i}
= \frac{13{,}500}{100} + \frac{10{,}100}{80} + \frac{8{,}000}{60}
\approx 394.58\text{ hours},
$$

whereas total available capacity is $6\times 8=48$ hours. The genetic algorithm therefore searches the penalized objective, which trades off cost components, rather than a set of capacity-feasible schedules. This distinction is essential when interpreting the returned schedule and cost.

### Genetic algorithm steps

1. **Chromosome encoding:** store production quantities in a product-by-period matrix.
2. **Population initialization:** generate varied schedules and repair each product's total production so that its cumulative production covers its horizon demand.
3. **Fitness evaluation:** calculate the modeled cost and use its negative as fitness, so a higher fitness means a lower cost.
4. **Selection:** use tournament selection to favor chromosomes with higher fitness.
5. **Crossover:** use uniform crossover to exchange production genes between two parents.
6. **Mutation:** randomly perturb individual production quantities and repair horizon-total production.
7. **Evolution:** preserve elite chromosomes and repeat selection and reproduction for the specified number of generations.

### Interpreting the results

The returned schedule is the best chromosome encountered by this stochastic search, not a guaranteed optimum. Review its capacity-violation penalties and period utilization alongside setup, holding, and shortage costs. The convergence plot tracks best and mean population costs; the diversity plot shows the standard deviation of fitness values, not a measure of schedule feasibility. Results can vary between runs because no random seed is imposed.

In [ ]:
from dataclasses import dataclass
import random
import time
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# These penalties keep unmet demand and capacity violations visible in the objective.
SHORTAGE_PENALTY_PER_UNIT = 100.0
CAPACITY_PENALTY_PER_HOUR = 1000.0

plt.style.use("default")
sns.set_palette("husl")

## 1. Define the instance

For product $i$ and period $t$, let $d_{it}$ be demand, $s_i$ the setup cost, $h_i$ the holding cost per unit per period, and $r_i$ the production rate in units per hour. The shared capacity in each period is $H$ hours. The demand matrix has one row per product and one column per period.

The data class checks that these arrays match the declared numbers of products and periods. The values below define the concrete three-product, six-period instance.

In [ ]:
@dataclass
class GAParameters:
    """Problem data required to evaluate a production schedule."""

    num_products: int
    num_periods: int
    demand_matrix: List[List[float]]
    setup_costs: List[float]
    holding_costs: List[float]
    production_rates: List[float]
    capacity_hours: float

    def __post_init__(self) -> None:
        """Check dimensions and basic parameter domains."""
        if self.num_products <= 0 or self.num_periods <= 0:
            raise ValueError("The number of products and periods must be positive.")
        if len(self.demand_matrix) != self.num_products:
            raise ValueError("Demand matrix rows must match the number of products.")
        if any(len(row) != self.num_periods for row in self.demand_matrix):
            raise ValueError("Demand matrix columns must match the number of periods.")
        if len(self.setup_costs) != self.num_products:
            raise ValueError("Setup costs must have one value per product.")
        if len(self.holding_costs) != self.num_products:
            raise ValueError("Holding costs must have one value per product.")
        if len(self.production_rates) != self.num_products:
            raise ValueError("Production rates must have one value per product.")
        if any(value < 0 for row in self.demand_matrix for value in row):
            raise ValueError("Demand values must be non-negative.")
        if any(value < 0 for value in self.setup_costs + self.holding_costs):
            raise ValueError("Setup and holding costs must be non-negative.")
        if any(value <= 0 for value in self.production_rates):
            raise ValueError("Production rates must be positive.")
        if self.capacity_hours <= 0:
            raise ValueError("Period capacity must be positive.")


demand_matrix = [
    [2300, 2100, 2400, 2200, 2400, 2100],
    [1800, 1600, 1750, 1650, 1700, 1600],
    [1350, 1250, 1450, 1300, 1400, 1250],
]

ga_params = GAParameters(
    num_products=3,
    num_periods=6,
    demand_matrix=demand_matrix,
    setup_costs=[500, 400, 600],
    holding_costs=[2.0, 1.5, 2.5],
    production_rates=[100, 80, 60],
    capacity_hours=8.0,
)

total_demand = sum(sum(row) for row in ga_params.demand_matrix)
required_hours = sum(
    sum(ga_params.demand_matrix[product]) / ga_params.production_rates[product]
    for product in range(ga_params.num_products)
)
total_available_hours = ga_params.num_periods * ga_params.capacity_hours

print("Genetic Algorithm Problem Parameters")
print(f"Number of products: {ga_params.num_products}")
print(f"Number of periods: {ga_params.num_periods}")
print(f"Shared capacity: {ga_params.capacity_hours:.1f} hours per period")
print(f"Setup costs: ${', '.join(map(str, ga_params.setup_costs))}")
print(f"Holding costs: ${', '.join(map(str, ga_params.holding_costs))} per unit per period")
print(f"Production rates: {', '.join(map(str, ga_params.production_rates))} units per hour")
print("\nDemand matrix:")
for product, product_demand in enumerate(ga_params.demand_matrix, start=1):
    print(f"  Product {product}: {product_demand}")
print(f"\nTotal demand: {total_demand:,} units")
print(f"Required production time: {required_hours:.2f} hours")
print(f"Available production time: {total_available_hours:.2f} hours")
print("A capacity-feasible schedule cannot meet all demand for this instance.")

## 2. Represent and evaluate schedules

A **chromosome** is a candidate production schedule. Its gene $q_{it}\geq 0$ is the quantity of product $i$ made in period $t$. Inventory follows the balance equation

$$
I_{it}=I_{i,t-1}+q_{it}-d_{it}, \qquad I_{i0}=0.
$$

Positive $I_{it}$ is inventory carried at the end of a period; negative $I_{it}$ is cumulative unmet demand. A setup is charged when $q_{it}>0$. Period production time is $\sum_i q_{it}/r_i$.

The implemented cost is

$$
C(q)=\sum_{i,t}\left[s_i\,\mathbf{1}(q_{it}>0)
+h_i\max(I_{it},0)
+100\max(-I_{it},0)\right]
+1000\sum_t\max\left(\sum_i\frac{q_{it}}{r_i}-H,0\right).
$$

Here $\mathbf{1}(q_{it}>0)$ equals one when production is positive and zero otherwise. The last two terms are penalty costs per unit of shortage and per excess production hour. The chromosome's fitness is $-C(q)$; maximizing fitness is therefore equivalent to minimizing this penalized cost. These penalties do not turn the capacity condition into a hard constraint.

The repair step enforces non-negative production and ensures that total production for each product over the horizon is at least its total demand. It does not guarantee that demand is met in every period, or that shared period capacity is respected.

In [ ]:
class Chromosome:
    """A production schedule and its evaluated cost information."""

    def __init__(self, genes: List[List[float]], params: GAParameters):
        if len(genes) != params.num_products:
            raise ValueError("Chromosome rows must match the number of products.")
        if any(len(row) != params.num_periods for row in genes):
            raise ValueError("Chromosome columns must match the number of periods.")
        self.genes = [[float(value) for value in row] for row in genes]
        self.params = params
        self.fitness = None
        self.setup_schedule = None
        self.inventory_levels = None
        self.cost_components = None

    def calculate_fitness(self) -> float:
        """Calculate negative penalized cost so higher fitness is better."""
        total_setup_cost = 0.0
        total_holding_cost = 0.0
        total_shortage_penalty = 0.0
        total_capacity_penalty = 0.0
        setup_schedule = []
        inventory_matrix = [
            [0.0] * self.params.num_periods
            for _ in range(self.params.num_products)
        ]

        for product in range(self.params.num_products):
            cumulative_inventory = 0.0
            product_setups = []

            for period in range(self.params.num_periods):
                production = self.genes[product][period]
                demand = self.params.demand_matrix[product][period]

                if production > 0:
                    total_setup_cost += self.params.setup_costs[product]
                    product_setups.append(period + 1)

                cumulative_inventory += production - demand
                inventory_matrix[product][period] = cumulative_inventory

                if cumulative_inventory > 0:
                    total_holding_cost += (
                        cumulative_inventory * self.params.holding_costs[product]
                    )
                elif cumulative_inventory < 0:
                    total_shortage_penalty += (
                        -cumulative_inventory * SHORTAGE_PENALTY_PER_UNIT
                    )

            setup_schedule.append(product_setups)

        for period in range(self.params.num_periods):
            production_hours = sum(
                self.genes[product][period] / self.params.production_rates[product]
                for product in range(self.params.num_products)
            )
            excess_hours = max(0.0, production_hours - self.params.capacity_hours)
            total_capacity_penalty += (
                excess_hours * CAPACITY_PENALTY_PER_HOUR
            )

        total_cost = (
            total_setup_cost
            + total_holding_cost
            + total_shortage_penalty
            + total_capacity_penalty
        )
        self.fitness = -total_cost
        self.setup_schedule = setup_schedule
        self.inventory_levels = inventory_matrix
        self.cost_components = {
            "setup": total_setup_cost,
            "holding": total_holding_cost,
            "shortage_penalty": total_shortage_penalty,
            "capacity_penalty": total_capacity_penalty,
        }
        return self.fitness

    def repair_genes(self) -> None:
        """Keep production non-negative and cover each product's horizon demand."""
        for product in range(self.params.num_products):
            self.genes[product] = [max(0.0, value) for value in self.genes[product]]
            total_production = sum(self.genes[product])
            total_demand = sum(self.params.demand_matrix[product])
            if total_production < total_demand:
                # Adding the remaining quantity in the final period repairs the
                # horizon total without changing the chromosome representation.
                self.genes[product][-1] += total_demand - total_production


def initialize_population(
    params: GAParameters,
    population_size: int,
) -> List[Chromosome]:
    """Create and evaluate a population of randomly initialized schedules."""
    if population_size <= 0:
        raise ValueError("Population size must be positive.")

    population = []
    for _ in range(population_size):
        genes = []
        for product in range(params.num_products):
            product_genes = []
            for period in range(params.num_periods):
                demand = params.demand_matrix[product][period]
                # This bounds a sampled gene; it is not a shared-capacity
                # feasibility check across products in the same period.
                max_production = min(
                    demand * 2,
                    params.capacity_hours * params.production_rates[product],
                )
                product_genes.append(random.uniform(0.0, max_production))
            genes.append(product_genes)

        chromosome = Chromosome(genes, params)
        chromosome.repair_genes()
        chromosome.calculate_fitness()
        population.append(chromosome)

    return population


test_population = initialize_population(ga_params, 5)
print(f"Initialized test population with {len(test_population)} chromosomes")
print(f"First chromosome fitness: {test_population[0].fitness:.2f}")
print(
    "First chromosome gene matrix shape: "
    f"{len(test_population[0].genes)} products x "
    f"{len(test_population[0].genes[0])} periods"
)

## 3. Selection, crossover, and mutation

Tournament selection samples a small group of chromosomes and chooses its highest-fitness member. This gives better schedules a greater chance of reproduction while retaining a chance for other schedules to be selected. Uniform crossover independently chooses which parent supplies each product-period gene. Mutation perturbs selected genes by a fraction of that product's period demand; repair then restores non-negative production and sufficient total horizon production.

These operators are the genetic algorithm's search mechanism: selection favors promising schedules, crossover recombines their schedule components, and mutation introduces variation. They do not guarantee feasibility or improvement for each individual offspring.

In [ ]:
def tournament_selection(
    population: List[Chromosome],
    tournament_size: int = 3,
) -> Chromosome:
    """Return the highest-fitness member of a randomly sampled tournament."""
    if not population:
        raise ValueError("Population must not be empty.")
    tournament = random.sample(population, min(tournament_size, len(population)))
    return max(tournament, key=lambda chromosome: chromosome.fitness)


def crossover(
    parent1: Chromosome,
    parent2: Chromosome,
    crossover_rate: float = 0.8,
) -> Tuple[Chromosome, Chromosome]:
    """Apply uniform crossover and return two offspring chromosomes."""
    if parent1.params is not parent2.params:
        raise ValueError("Both parents must refer to the same problem parameters.")
    if not 0.0 <= crossover_rate <= 1.0:
        raise ValueError("Crossover rate must be between zero and one.")

    if random.random() >= crossover_rate:
        return (
            Chromosome(parent1.genes, parent1.params),
            Chromosome(parent2.genes, parent2.params),
        )

    genes1 = []
    genes2 = []
    for product in range(parent1.params.num_products):
        product_genes1 = []
        product_genes2 = []
        for period in range(parent1.params.num_periods):
            if random.random() < 0.5:
                product_genes1.append(parent1.genes[product][period])
                product_genes2.append(parent2.genes[product][period])
            else:
                product_genes1.append(parent2.genes[product][period])
                product_genes2.append(parent1.genes[product][period])
        genes1.append(product_genes1)
        genes2.append(product_genes2)

    return (
        Chromosome(genes1, parent1.params),
        Chromosome(genes2, parent2.params),
    )


def mutation(
    chromosome: Chromosome,
    mutation_rate: float = 0.1,
    mutation_strength: float = 0.2,
) -> Chromosome:
    """Perturb selected genes, then repair the resulting schedule."""
    if not 0.0 <= mutation_rate <= 1.0:
        raise ValueError("Mutation rate must be between zero and one.")
    if mutation_strength < 0:
        raise ValueError("Mutation strength must be non-negative.")

    mutated_genes = []
    for product in range(chromosome.params.num_products):
        product_genes = []
        for period in range(chromosome.params.num_periods):
            current_value = chromosome.genes[product][period]
            if random.random() < mutation_rate:
                demand = chromosome.params.demand_matrix[product][period]
                max_change = demand * mutation_strength
                change = random.uniform(-max_change, max_change)
                product_genes.append(max(0.0, current_value + change))
            else:
                product_genes.append(current_value)
        mutated_genes.append(product_genes)

    mutated_chromosome = Chromosome(mutated_genes, chromosome.params)
    mutated_chromosome.repair_genes()
    return mutated_chromosome


offspring1, offspring2 = crossover(test_population[0], test_population[1])
mutated_offspring = mutation(offspring1)

print("Genetic Operator Demonstration")
print(f"Parent 1 fitness: {test_population[0].fitness:.2f}")
print(f"Parent 2 fitness: {test_population[1].fitness:.2f}")
print(f"Offspring 1 fitness: {offspring1.calculate_fitness():.2f}")
print(f"Offspring 2 fitness: {offspring2.calculate_fitness():.2f}")
print(f"Mutated offspring fitness: {mutated_offspring.calculate_fitness():.2f}")

## 4. Evolve a population

At each generation, the algorithm preserves the highest-fitness chromosomes (elitism), selects parents by tournament, applies crossover and mutation, and evaluates the resulting schedules. The best fitness and mean fitness are recorded for the initial population and after every generation. Population diversity is the standard deviation of fitness values measured before reproduction in each generation.

If the population size is $P$, the number of generations is $G$, there are $n$ products, and there are $T$ periods, evaluating one population costs $O(PnT)$. Across the run, the dominant evaluation work is $O(GPnT)$. This describes computational effort; it does not imply that the search finds a global optimum.

In [ ]:
def genetic_algorithm(
    params: GAParameters,
    population_size: int = 50,
    generations: int = 300,
    crossover_rate: float = 0.8,
    mutation_rate: float = 0.1,
    elitism_count: int = 2,
) -> Dict:
    """Search for a low-cost production schedule using a genetic algorithm."""
    if population_size < 2:
        raise ValueError("Population size must be at least two.")
    if generations < 0:
        raise ValueError("Generation count must be non-negative.")
    if not 0 <= elitism_count <= population_size:
        raise ValueError("Elitism count must be between zero and population size.")
    if not 0.0 <= crossover_rate <= 1.0:
        raise ValueError("Crossover rate must be between zero and one.")
    if not 0.0 <= mutation_rate <= 1.0:
        raise ValueError("Mutation rate must be between zero and one.")

    population = initialize_population(params, population_size)
    best_chromosome = max(population, key=lambda chromosome: chromosome.fitness)
    best_fitness_history = [best_chromosome.fitness]
    avg_fitness_history = [
        float(np.mean([chromosome.fitness for chromosome in population]))
    ]
    diversity_history = []

    for generation in range(generations):
        fitness_values = [chromosome.fitness for chromosome in population]
        diversity = float(np.std(fitness_values))
        diversity_history.append(diversity)

        sorted_population = sorted(
            population,
            key=lambda chromosome: chromosome.fitness,
            reverse=True,
        )
        new_population = sorted_population[:elitism_count]

        while len(new_population) < population_size:
            parent1 = tournament_selection(population)
            parent2 = tournament_selection(population)
            child1, child2 = crossover(parent1, parent2, crossover_rate)
            child1 = mutation(child1, mutation_rate)
            child2 = mutation(child2, mutation_rate)
            child1.calculate_fitness()
            child2.calculate_fitness()
            new_population.extend([child1, child2])

        population = new_population[:population_size]
        current_best = max(population, key=lambda chromosome: chromosome.fitness)
        if current_best.fitness > best_chromosome.fitness:
            best_chromosome = current_best

        best_fitness_history.append(best_chromosome.fitness)
        avg_fitness_history.append(
            float(np.mean([chromosome.fitness for chromosome in population]))
        )

        if (generation + 1) % 50 == 0:
            print(
                f"Generation {generation + 1:3d}: "
                f"Best cost = {-best_chromosome.fitness:.2f}, "
                f"Mean cost = {-avg_fitness_history[-1]:.2f}, "
                f"Fitness diversity = {diversity:.2f}"
            )

    best_chromosome.calculate_fitness()
    return {
        "best_chromosome": best_chromosome,
        "best_fitness": best_chromosome.fitness,
        "best_cost": -best_chromosome.fitness,
        "best_schedule": best_chromosome.genes,
        "setup_schedule": best_chromosome.setup_schedule,
        "inventory_levels": best_chromosome.inventory_levels,
        "cost_components": best_chromosome.cost_components,
        "best_fitness_history": best_fitness_history,
        "avg_fitness_history": avg_fitness_history,
        "diversity_history": diversity_history,
        "generations": generations,
        "population_size": population_size,
    }


print("Running the genetic algorithm for the multi-product POQ problem...")
start_time = time.time()
ga_results = genetic_algorithm(
    ga_params,
    population_size=40,
    generations=300,
    crossover_rate=0.8,
    mutation_rate=0.1,
    elitism_count=2,
)
execution_time = time.time() - start_time

print(f"\nGenetic algorithm completed in {execution_time:.2f} seconds")
print("\nGenetic Algorithm Search Results")
print(f"Best penalized cost found: ${ga_results['best_cost']:,.2f}")
print(f"Best fitness: {ga_results['best_fitness']:.2f}")
print(f"Generations: {ga_results['generations']}")
print(f"Population size: {ga_results['population_size']}")
print("The best schedule is a search result, not a guaranteed optimum.")

## 5. Inspect the best schedule found

The table reports production, demand, and end-of-period cumulative inventory for each product and period. Setup periods are those with positive production. The cost breakdown separates setup and holding costs from shortage and capacity penalties, so a low total cannot be mistaken for a capacity-feasible plan. Period utilization is calculated from the unrounded production quantities and may exceed 100% when the capacity penalty is active.

In [ ]:
best_chromosome = ga_results["best_chromosome"]
assert best_chromosome.calculate_fitness() is not None
assert np.isclose(-best_chromosome.fitness, ga_results["best_cost"])
assert np.isclose(
    sum(ga_results["cost_components"].values()),
    ga_results["best_cost"],
)

print("\nBest Production Schedule Found")
for product in range(ga_params.num_products):
    print(f"\nProduct {product + 1} (setup cost: ${ga_params.setup_costs[product]:,.2f})")
    print(f"  Setup periods: {ga_results['setup_schedule'][product]}")
    for period in range(ga_params.num_periods):
        production = ga_results["best_schedule"][product][period]
        demand = ga_params.demand_matrix[product][period]
        inventory = ga_results["inventory_levels"][product][period]
        status = "PRODUCE" if production > 0 else "IDLE"
        print(
            f"  Period {period + 1}: {production:9.2f} units ({status}), "
            f"demand: {demand:6.0f}, end inventory: {inventory:10.2f}"
        )

print("\nCost Breakdown")
for component, cost in ga_results["cost_components"].items():
    label = component.replace("_", " ").title()
    print(f"{label}: ${cost:,.2f}")
print(f"Total penalized cost: ${ga_results['best_cost']:,.2f}")

print("\nCapacity Use by Period")
for period in range(ga_params.num_periods):
    production_hours = sum(
        ga_results["best_schedule"][product][period]
        / ga_params.production_rates[product]
        for product in range(ga_params.num_products)
    )
    utilization = production_hours / ga_params.capacity_hours * 100
    print(
        f"Period {period + 1}: {production_hours:.2f} hours, "
        f"{utilization:.1f}% of the stated capacity"
    )

print(
    "\nCapacity-feasible across all periods: "
    f"{all(sum(ga_results['best_schedule'][product][period] / ga_params.production_rates[product] for product in range(ga_params.num_products)) <= ga_params.capacity_hours + 1e-9 for period in range(ga_params.num_periods))}"
)

## 6. Visualize search and schedule characteristics

The convergence curves show cost rather than fitness, so lower values are better. The best-cost history has one entry for the initial population plus one entry per generation. Diversity is measured once per generation before reproduction, so its horizontal axis starts at generation 1. The heatmap shows production quantities, and the final panel counts setups by product.

In [ ]:
def create_ga_visualizations(ga_results: Dict, params: GAParameters):
    """Plot convergence, population diversity, production, and setup counts."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Genetic Algorithm Multi-Product POQ Analysis",
        fontsize=16,
        fontweight="bold",
    )

    history_generations = range(len(ga_results["best_fitness_history"]))
    best_costs = [-value for value in ga_results["best_fitness_history"]]
    mean_costs = [-value for value in ga_results["avg_fitness_history"]]
    ax1.plot(history_generations, best_costs, linewidth=2, label="Best cost")
    ax1.plot(
        history_generations,
        mean_costs,
        linestyle="--",
        linewidth=1,
        alpha=0.8,
        label="Mean population cost",
    )
    ax1.set_xlabel("Generation (0 is the initial population)")
    ax1.set_ylabel("Penalized cost ($)")
    ax1.set_title("Search Progress")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    diversity_generations = range(1, len(ga_results["diversity_history"]) + 1)
    ax2.plot(
        diversity_generations,
        ga_results["diversity_history"],
        color="green",
        linewidth=2,
    )
    ax2.set_xlabel("Generation")
    ax2.set_ylabel("Fitness standard deviation")
    ax2.set_title("Population Fitness Diversity")
    ax2.grid(True, alpha=0.3)

    production_matrix = np.asarray(ga_results["best_schedule"], dtype=float)
    image = ax3.imshow(production_matrix, cmap="YlOrRd", aspect="auto")
    for product in range(params.num_products):
        for period in range(params.num_periods):
            ax3.text(
                period,
                product,
                f"{production_matrix[product, period]:.0f}",
                ha="center",
                va="center",
                color="black",
                fontsize=8,
            )
    ax3.set_xlabel("Period")
    ax3.set_ylabel("Product")
    ax3.set_title("Production Schedule")
    ax3.set_xticks(range(params.num_periods))
    ax3.set_xticklabels([f"Period {period + 1}" for period in range(params.num_periods)])
    ax3.set_yticks(range(params.num_products))
    ax3.set_yticklabels([f"Product {product + 1}" for product in range(params.num_products)])
    fig.colorbar(image, ax=ax3, label="Production quantity (units)")

    setup_counts = [len(periods) for periods in ga_results["setup_schedule"]]
    product_labels = [f"Product {product + 1}" for product in range(params.num_products)]
    bars = ax4.bar(product_labels, setup_counts, alpha=0.8)
    ax4.set_xlabel("Product")
    ax4.set_ylabel("Number of setups")
    ax4.set_title("Setup Frequency by Product")
    ax4.grid(True, alpha=0.3, axis="y")
    for bar, count in zip(bars, setup_counts):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(count),
            ha="center",
            va="bottom",
        )

    fig.tight_layout()
    plt.show()
    return fig


fig = create_ga_visualizations(ga_results, ga_params)

## 7. Compare with an every-other-period heuristic

A comparison is meaningful only when both schedules are evaluated with the same cost definition. The heuristic produces enough to cover the current and next period's demand on every other period; it is a reference schedule, not a second optimization algorithm. Its evaluation includes the same setup, holding, shortage, and capacity-violation terms as the genetic algorithm. A positive cost improvement means the genetic algorithm's best penalized cost is lower than the heuristic's.

In [ ]:
def compare_with_heuristic(ga_results: Dict, params: GAParameters) -> Dict:
    """Evaluate an every-other-period schedule using the shared cost model."""
    heuristic_schedule = []

    for product in range(params.num_products):
        product_schedule = []
        cumulative_inventory = 0.0

        for period in range(params.num_periods):
            if period % 2 == 0:
                demand_this_period = params.demand_matrix[product][period]
                demand_next_period = (
                    params.demand_matrix[product][period + 1]
                    if period + 1 < params.num_periods
                    else 0
                )
                production = max(
                    0.0,
                    demand_this_period + demand_next_period - cumulative_inventory,
                )
            else:
                production = 0.0

            product_schedule.append(production)
            cumulative_inventory += production - params.demand_matrix[product][period]

        heuristic_schedule.append(product_schedule)

    heuristic_chromosome = Chromosome(heuristic_schedule, params)
    heuristic_chromosome.calculate_fitness()
    heuristic_cost = -heuristic_chromosome.fitness
    ga_cost = ga_results["best_cost"]
    cost_improvement = heuristic_cost - ga_cost
    cost_improvement_pct = (
        100 * cost_improvement / heuristic_cost if heuristic_cost != 0 else 0.0
    )

    return {
        "heuristic_cost": heuristic_cost,
        "ga_cost": ga_cost,
        "cost_improvement": cost_improvement,
        "cost_improvement_pct": cost_improvement_pct,
        "heuristic_schedule": heuristic_schedule,
        "heuristic_setups": heuristic_chromosome.setup_schedule,
        "heuristic_cost_components": heuristic_chromosome.cost_components,
    }


comparison = compare_with_heuristic(ga_results, ga_params)
print("\nComparison Under the Shared Penalized Cost Model")
print(f"Genetic algorithm cost: ${comparison['ga_cost']:,.2f}")
print(f"Every-other-period heuristic cost: ${comparison['heuristic_cost']:,.2f}")
print(
    f"Cost improvement (heuristic cost minus GA cost): "
    f"${comparison['cost_improvement']:,.2f} "
    f"({comparison['cost_improvement_pct']:.1f}%)"
)
print(
    "GA setup count: "
    f"{sum(len(periods) for periods in ga_results['setup_schedule'])}"
)
print(
    "Heuristic setup count: "
    f"{sum(len(periods) for periods in comparison['heuristic_setups'])}"
)
if ga_results["diversity_history"]:
    print(f"Final recorded fitness diversity: {ga_results['diversity_history'][-1]:.2f}")
best_generation = int(np.argmax(ga_results["best_fitness_history"]))
print(
    "Best-so-far schedule first recorded at generation "
    f"{best_generation} (generation 0 is the initial population)."
)

## 8. Strengths and limitations of the genetic algorithm

A population-based search can explore many candidate schedules and can be adapted by changing the chromosome evaluation and genetic operators. Its stochastic operators may discover useful combinations of production quantities that are not apparent from a simple rule. These properties do not establish that the method is globally optimal, scalable for every instance, or superior to another method.

The result depends on population size, generation count, crossover and mutation rates, initialization, and random draws. Penalty-based constraint handling also allows infeasible candidates; the penalty weights affect the trade-off between cost components and constraint violations. For this instance, the stated total demand requires substantially more production time than the stated total capacity, so no schedule can satisfy both full demand and all capacity limits. The reported best chromosome must therefore be interpreted as a low-cost candidate under the specified penalty model, not as a feasible production plan.

## Summary

This notebook encodes multi-product production schedules as chromosomes and uses tournament selection, uniform crossover, mutation, repair, and elitism to search for a low-cost schedule. The objective includes setup costs, holding costs, shortage penalties, and penalties for exceeding shared period capacity. Fitness is the negative of this total cost.

Interpret the displayed result as the best schedule encountered in a stochastic search. The method does not guarantee global optimality, and the capacity penalty does not enforce a hard capacity limit. In this instance, required production time is approximately 394.58 hours while total available capacity is 48 hours, so meeting all demand within capacity is impossible under the stated data. The cost breakdown and utilization report are therefore necessary parts of interpreting the schedule; a low penalized cost alone does not establish feasibility.

The convergence and diversity plots describe search behavior, while the heuristic comparison evaluates a simple reference schedule under the same cost model. Neither the search histories nor the comparison constitute a proof of optimality.